# ANRF AISEHack 2.0 - Polymer Property Prediction
**Approach:** RDKit descriptors + Morgan fingerprints + MACCS keys → LightGBM/XGBoost ensemble with Optuna tuning

**Targets:** Tg (glass transition temperature, °C) and Egc (chain band gap, eV) — predicted separately.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, MACCSkeys

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

import xgboost as xgb
import lightgbm as lgb

SEED = 42
np.random.seed(SEED)
print('Imports done.')

## 1. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/aisehack-2-0/train.csv')
test  = pd.read_csv('/kaggle/input/aisehack-2-0/test.csv')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print()
print('Target type distribution (train):')
print(train['target_type'].value_counts())
print()
print('Target type distribution (test):')
print(test['target_type'].value_counts())
print()
train.head(3)

In [ ]:
# Split by property
train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

print(f'Train Tg: {len(train_tg)}, Train Egc: {len(train_egc)}')
print(f'Test  Tg: {len(test_tg)},  Test  Egc: {len(test_egc)}')
print(f'Tg range: [{train_tg.target.min():.1f}, {train_tg.target.max():.1f}]')
print(f'Egc range: [{train_egc.target.min():.3f}, {train_egc.target.max():.3f}]')

## 2. Feature Engineering

Three complementary feature sets:
- **RDKit 2D descriptors** (~210 features): physicochemical properties (MW, logP, TPSA, ring counts, etc.)
- **Morgan fingerprints** (radius=2, 2048 bits): circular substructure encoding, great for capturing local chemistry
- **MACCS keys** (166 bits): standardised structural keys used widely in cheminformatics

In [ ]:
def smiles_to_mol(smiles):
    """Parse SMILES, return None if invalid."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol
    except Exception:
        return None


def get_rdkit_descriptors(smiles_list):
    """Compute all RDKit 2D descriptors. Returns DataFrame."""
    desc_names = [n for n, _ in Descriptors.descList]
    rows = []
    for smi in smiles_list:
        mol = smiles_to_mol(smi)
        if mol is None:
            rows.append([np.nan] * len(desc_names))
        else:
            try:
                vals = Descriptors.CalcMolDescriptors(mol)
                rows.append(list(vals.values()))
            except Exception:
                rows.append([np.nan] * len(desc_names))
    return pd.DataFrame(rows, columns=desc_names)


def get_morgan_fingerprints(smiles_list, radius=2, n_bits=2048):
    """Compute Morgan (ECFP) fingerprints. Returns DataFrame."""
    rows = []
    cols = [f'morgan_{i}' for i in range(n_bits)]
    for smi in smiles_list:
        mol = smiles_to_mol(smi)
        if mol is None:
            rows.append([0] * n_bits)
        else:
            try:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
                rows.append(list(fp))
            except Exception:
                rows.append([0] * n_bits)
    return pd.DataFrame(rows, columns=cols, dtype=np.uint8)


def get_maccs_keys(smiles_list):
    """Compute MACCS keys (166 bits). Returns DataFrame."""
    rows = []
    cols = [f'maccs_{i}' for i in range(167)]  # bits 0..166, bit 0 unused
    for smi in smiles_list:
        mol = smiles_to_mol(smi)
        if mol is None:
            rows.append([0] * 167)
        else:
            try:
                fp = MACCSkeys.GenMACCSKeys(mol)
                rows.append(list(fp))
            except Exception:
                rows.append([0] * 167)
    return pd.DataFrame(rows, columns=cols, dtype=np.uint8)


def build_features(df, fit_imputer_scaler=None):
    """
    Build combined feature matrix from SMILES column.
    fit_imputer_scaler: if None, fit and return new ones;
                        else use provided (imputer, scaler) tuple.
    Returns: X_scaled (np array), (imputer, scaler)
    """
    smiles = df['smiles'].tolist()
    print('  Computing RDKit descriptors...')
    rdkit_df  = get_rdkit_descriptors(smiles)
    print('  Computing Morgan fingerprints...')
    morgan_df = get_morgan_fingerprints(smiles)
    print('  Computing MACCS keys...')
    maccs_df  = get_maccs_keys(smiles)

    # Combine
    X = pd.concat([rdkit_df, morgan_df, maccs_df], axis=1)

    # Replace inf
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.astype(float)

    print(f'  Total features: {X.shape[1]}')

    if fit_imputer_scaler is None:
        # Drop columns where >80% are NaN
        thresh = int(0.2 * len(X))
        X = X.dropna(axis=1, thresh=thresh)
        good_cols = X.columns.tolist()

        # Drop zero-variance columns
        var = X.var()
        X = X.loc[:, var > 0]
        good_cols = X.columns.tolist()

        imputer = SimpleImputer(strategy='median')
        X_imp = imputer.fit_transform(X)

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_imp)

        return X_scaled, (imputer, scaler, good_cols)
    else:
        imputer, scaler, good_cols = fit_imputer_scaler
        # Keep only training columns (some test SMILES might differ)
        X = X.reindex(columns=good_cols, fill_value=np.nan)
        X_imp = imputer.transform(X)
        X_scaled = scaler.transform(X_imp)
        return X_scaled, fit_imputer_scaler


print('Feature functions defined.')

In [ ]:
print('Building features for Tg (train)...')
X_tg_train, tg_preprocessor = build_features(train_tg)
y_tg = train_tg['target'].values

print('\nBuilding features for Egc (train)...')
X_egc_train, egc_preprocessor = build_features(train_egc)
y_egc = train_egc['target'].values

print(f'\nX_tg shape:  {X_tg_train.shape}')
print(f'X_egc shape: {X_egc_train.shape}')

## 3. Model Training

Strategy: **5-fold CV ensemble** of LightGBM + XGBoost, averaged predictions.

LightGBM and XGBoost are complementary — they differ in their tree-building algorithms (leaf-wise vs depth-wise), so their errors partially cancel when averaged.

In [ ]:
def get_lgbm_params(target_type):
    """Sensible LightGBM params per property."""
    base = dict(
        objective='regression',
        metric='rmse',
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.4,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    if target_type == 'tg':
        base['num_leaves'] = 127
        base['colsample_bytree'] = 0.5
    return base


def get_xgb_params(target_type):
    """Sensible XGBoost params per property."""
    base = dict(
        objective='reg:squarederror',
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.4,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
    )
    if target_type == 'egc':
        base['max_depth'] = 5
    return base


def train_cv_ensemble(X_train, y_train, X_test, target_type, n_splits=5):
    """
    5-fold CV ensemble of LightGBM + XGBoost.
    Returns: oof predictions, test predictions, per-fold R² scores.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    oof_lgbm = np.zeros(len(X_train))
    oof_xgb  = np.zeros(len(X_train))
    test_lgbm = np.zeros(len(X_test))
    test_xgb  = np.zeros(len(X_test))

    lgbm_params = get_lgbm_params(target_type)
    xgb_params  = get_xgb_params(target_type)

    fold_r2_lgbm, fold_r2_xgb = [], []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        # --- LightGBM ---
        model_lgbm = lgb.LGBMRegressor(**lgbm_params)
        model_lgbm.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(period=-1)]
        )
        oof_lgbm[val_idx] = model_lgbm.predict(X_val)
        test_lgbm += model_lgbm.predict(X_test) / n_splits
        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        fold_r2_lgbm.append(r2_l)

        # --- XGBoost ---
        model_xgb = xgb.XGBRegressor(**xgb_params)
        model_xgb.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=100
        )
        oof_xgb[val_idx] = model_xgb.predict(X_val)
        test_xgb += model_xgb.predict(X_test) / n_splits
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        fold_r2_xgb.append(r2_x)

        print(f'  Fold {fold}: LGBM R²={r2_l:.4f} | XGB R²={r2_x:.4f}')

    # Simple average ensemble
    oof_ensemble  = (oof_lgbm + oof_xgb) / 2
    test_ensemble = (test_lgbm + test_xgb) / 2

    oof_r2 = r2_score(y_train, oof_ensemble)
    print(f'  OOF R² (ensemble): {oof_r2:.4f}')
    print(f'  LGBM mean fold R²: {np.mean(fold_r2_lgbm):.4f} ± {np.std(fold_r2_lgbm):.4f}')
    print(f'  XGB  mean fold R²: {np.mean(fold_r2_xgb):.4f} ± {np.std(fold_r2_xgb):.4f}')

    return oof_ensemble, test_ensemble, oof_r2


print('Training functions defined.')

In [ ]:
print('Building test features for Tg...')
X_tg_test, _ = build_features(test_tg, fit_imputer_scaler=tg_preprocessor)

print('\nBuilding test features for Egc...')
X_egc_test, _ = build_features(test_egc, fit_imputer_scaler=egc_preprocessor)

In [ ]:
print('=' * 50)
print('Training Tg models...')
print('=' * 50)
oof_tg, pred_tg, r2_tg = train_cv_ensemble(
    X_tg_train, y_tg, X_tg_test, target_type='tg'
)

print()
print('=' * 50)
print('Training Egc models...')
print('=' * 50)
oof_egc, pred_egc, r2_egc = train_cv_ensemble(
    X_egc_train, y_egc, X_egc_test, target_type='egc'
)

print()
print('=' * 50)
mean_r2 = (r2_tg + r2_egc) / 2
print(f'OOF R² Tg:  {r2_tg:.4f}')
print(f'OOF R² Egc: {r2_egc:.4f}')
print(f'Mean OOF R²: {mean_r2:.4f}  ← competition metric proxy')
print('=' * 50)

## 4. Generate Submission

In [ ]:
test_tg_out  = test_tg[['id']].copy()
test_tg_out['target'] = pred_tg

test_egc_out = test_egc[['id']].copy()
test_egc_out['target'] = pred_egc

submission = pd.concat([test_tg_out, test_egc_out], axis=0).sort_values('id').reset_index(drop=True)

print('Submission shape:', submission.shape)
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved.')

## 5. Quick Sanity Checks

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_tg, oof_tg, alpha=0.3, s=10)
mn, mx = min(y_tg.min(), oof_tg.min()), max(y_tg.max(), oof_tg.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1)
axes[0].set_xlabel('True Tg (°C)')
axes[0].set_ylabel('Predicted Tg (°C)')
axes[0].set_title(f'Tg OOF — R² = {r2_tg:.4f}')

axes[1].scatter(y_egc, oof_egc, alpha=0.3, s=10, color='orange')
mn, mx = min(y_egc.min(), oof_egc.min()), max(y_egc.max(), oof_egc.max())
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=1)
axes[1].set_xlabel('True Egc (eV)')
axes[1].set_ylabel('Predicted Egc (eV)')
axes[1].set_title(f'Egc OOF — R² = {r2_egc:.4f}')

plt.tight_layout()
plt.savefig('oof_scatter.png', dpi=100)
plt.show()
print('Plot saved.')

In [ ]:
# Prediction distribution sanity check
print('Test Tg predictions:')
print(f'  Min: {pred_tg.min():.2f}  Max: {pred_tg.max():.2f}  Mean: {pred_tg.mean():.2f}')
print(f'  Train range: [{y_tg.min():.2f}, {y_tg.max():.2f}]')
print()
print('Test Egc predictions:')
print(f'  Min: {pred_egc.min():.4f}  Max: {pred_egc.max():.4f}  Mean: {pred_egc.mean():.4f}')
print(f'  Train range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

# Check for NaNs in submission
print()
print(f'NaNs in submission: {submission["target"].isna().sum()}')
print('All good!')